# Day 6 — Plotly Interactive Charts + Streamlit
**Date:** 26 March 2026 | **Phase 1 — Week 1** | **Roadmap: DS-AI-75D**

> Building interactive visualisations with Plotly and deploying
> a Streamlit data exploration app on the Titanic dataset.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly

print(f"Pandas:  {pd.__version__}")
print(f"Plotly:  {plotly.__version__}")

# Load local Titanic CSV
df = pd.read_csv(r"C:\DS-AI-75d\titanic.csv")
print(f"Dataset: {df.shape}")
print("Ready!")

Pandas:  2.3.3
Plotly:  6.6.0
Dataset: (891, 12)
Ready!


## 2. Plotly Express — Interactive Charts in One Line

### 2.1 Interactive Bar Chart — Survival Rate by Class

In [ ]:
# Plotly Express — beautiful interactive charts in 1-2 lines
# Hover over bars to see exact values

survival_by_class = df.groupby("Pclass")["Survived"].mean().reset_index()
survival_by_class.columns = ["Class", "Survival Rate"]
survival_by_class["Class"] = ["1st Class", "2nd Class", "3rd Class"]

fig = px.bar(
    survival_by_class,
    x="Class",
    y="Survival Rate",
    color="Survival Rate",
    color_continuous_scale="RdYlGn",
    title="Survival Rate by Passenger Class",
    text="Survival Rate",
)

fig.update_traces(texttemplate="%{text:.1%}", textposition="outside")
fig.update_layout(template="plotly_dark", height=450, showlegend=False)

fig.show()
print("Interactive bar chart done!")

Interactive bar chart done!


### 2.2 Interactive Scatter Plot — Age vs Fare

In [ ]:
# Interactive scatter — hover shows passenger details
fig = px.scatter(
    df.dropna(subset=["Age"]),
    x="Age",
    y="Fare",
    color="Survived",
    color_discrete_map={0: "#ef4444", 1: "#22c55e"},
    symbol="Sex",
    hover_data=["Name", "Pclass"],
    title="Age vs Fare — Coloured by Survival",
    labels={"Survived": "Survived", "Age": "Age", "Fare": "Fare (£)"},
    opacity=0.7,
)

fig.update_layout(template="plotly_dark", height=500, legend_title_text="Survived")

# Clip y-axis to remove extreme outliers visually
fig.update_yaxes(range=[-10, 300])

fig.show()
print("Interactive scatter done!")

Interactive scatter done!


### 2.3 Interactive Histogram — Age Distribution

In [ ]:
# Interactive histogram — click legend to show/hide groups
fig = px.histogram(
    df.dropna(subset=["Age"]),
    x="Age",
    color="Survived",
    color_discrete_map={0: "#ef4444", 1: "#22c55e"},
    barmode="overlay",
    nbins=30,
    opacity=0.7,
    title="Age Distribution — Survivors vs Non-Survivors",
    labels={"Survived": "Survived", "Age": "Age"},
)

fig.update_layout(template="plotly_dark", height=450, bargap=0.1)

# Add median lines
for survived, color, label in [(0, "#ef4444", "Died"), (1, "#22c55e", "Survived")]:
    median_age = df[df["Survived"] == survived]["Age"].median()
    fig.add_vline(
        x=median_age,
        line_dash="dash",
        line_color=color,
        annotation_text=f"{label} median: {median_age:.0f}",
        annotation_position="top right",
    )

fig.show()
print("Interactive histogram done!")

Interactive histogram done!


### 2.4 Interactive Box Plot — Fare by Class

In [ ]:
# Interactive boxplot — hover shows exact quartile values
fig = px.box(
    df,
    x="Pclass",
    y="Fare",
    color="Pclass",
    color_discrete_map={1: "#38bdf8", 2: "#818cf8", 3: "#f472b6"},
    title="Fare Distribution by Passenger Class",
    labels={"Pclass": "Passenger Class", "Fare": "Fare (£)"},
    points="outliers",  # Show outlier points
)

fig.update_layout(
    template="plotly_dark",
    height=450,
    showlegend=False,
    xaxis=dict(tickvals=[1, 2, 3], ticktext=["1st Class", "2nd Class", "3rd Class"]),
)

fig.update_yaxes(range=[-10, 300])
fig.show()
print("Interactive boxplot done!")

Interactive boxplot done!


### 2.5 Interactive Heatmap — Survival Rate by Class and Gender

In [ ]:
# Interactive heatmap — hover shows exact survival rates
pivot = df.groupby(["Pclass", "Sex"])["Survived"].mean().unstack()
pivot.index = ["1st Class", "2nd Class", "3rd Class"]

fig = px.imshow(
    pivot,
    color_continuous_scale="RdYlGn",
    title="Survival Rate by Class and Gender",
    text_auto=".1%",
    aspect="auto",
)

fig.update_layout(
    template="plotly_dark", height=400, coloraxis_colorbar_title="Survival Rate"
)

fig.show()
print("Interactive heatmap done!")

Interactive heatmap done!


## 3. Plotly Advanced — Subplots

### 3.1 Multi-Chart Dashboard in One Figure

In [ ]:
# Multiple interactive charts in one figure
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Survival by Class",
        "Survival by Gender",
        "Age Distribution",
        "Fare by Class",
    ),
)

# Chart 1 — Survival by Class
class_surv = df.groupby("Pclass")["Survived"].mean()
fig.add_trace(
    go.Bar(
        x=["1st", "2nd", "3rd"],
        y=class_surv.values,
        marker_color=["#38bdf8", "#818cf8", "#f472b6"],
        name="By Class",
    ),
    row=1,
    col=1,
)

# Chart 2 — Survival by Gender
sex_surv = df.groupby("Sex")["Survived"].mean()
fig.add_trace(
    go.Bar(
        x=sex_surv.index,
        y=sex_surv.values,
        marker_color=["#f472b6", "#38bdf8"],
        name="By Gender",
    ),
    row=1,
    col=2,
)

# Chart 3 — Age Distribution
fig.add_trace(
    go.Histogram(
        x=df[df["Survived"] == 1]["Age"].dropna(),
        name="Survived",
        marker_color="#22c55e",
        opacity=0.7,
        nbinsx=25,
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Histogram(
        x=df[df["Survived"] == 0]["Age"].dropna(),
        name="Died",
        marker_color="#ef4444",
        opacity=0.7,
        nbinsx=25,
    ),
    row=2,
    col=1,
)

# Chart 4 — Fare Boxplot
for cls, color in zip([1, 2, 3], ["#38bdf8", "#818cf8", "#f472b6"]):
    fig.add_trace(
        go.Box(
            y=df[df["Pclass"] == cls]["Fare"], name=f"Class {cls}", marker_color=color
        ),
        row=2,
        col=2,
    )

fig.update_layout(
    template="plotly_dark",
    height=700,
    title_text="Titanic Survival Dashboard",
    showlegend=False,
)

fig.show()
print("Dashboard done!")

Dashboard done!


## 4. Key Insights — Plotly vs Matplotlib

1. **Interactivity** — Plotly charts support hover, zoom, pan, click-to-hide by default
2. **One line charts** — px.bar(), px.scatter(), px.histogram() vs 10+ lines in Matplotlib
3. **hover_data** — show extra info on hover (Name, Pclass) without cluttering the chart
4. **color_discrete_map** — precise colour control per category value
5. **template='plotly_dark'** — professional dark theme in one parameter
6. **add_vline()** — add reference lines with annotations easily
7. **make_subplots** — combine multiple chart types in one interactive dashboard
